# WASB fine-tune SPARSE (Kaggle) — supervisión solo-centro, sin interpolar

El experimento decisivo y barato: entrenar WASB con los labels ralos reales
(supervisión SOLO del frame central, 3 frames de contexto, SIN targets inventados).
Ataca la causa del fracaso anterior (interpolación ruidosa).

## ⚙️ ANTES DE CORRER (panel derecho):
1. **Settings → Accelerator → GPU T4 x2**.
2. **Settings → Internet → ON**.
3. **Add Input → Dataset** con el video de brasil (`.mp4`).

Qué mirar: `BEFORE` (WASB preentrenado) vs las épocas. Si `acc@100` sube claro sobre
el baseline → WASB sirve con labels ralos. Si no → cerramos WASB con evidencia.

## 1) Setup

In [ ]:
import os, subprocess, glob
os.chdir('/kaggle/working')
if not os.path.exists('WASB-SBDT'):
    !git clone -q https://github.com/nttcom/WASB-SBDT.git
if not os.path.exists('ncf_event_tracker'):
    !git clone -q --branch events-model https://github.com/pipachiesa/ncf_event_tracker.git
!cd ncf_event_tracker && git pull -q origin events-model
!pip install -q hydra-core omegaconf gdown
W='/kaggle/working/wasb_soccer_best.pth.tar'
if not os.path.exists(W):
    import gdown; gdown.download(id='1pg0MpMtKZ6ziYEr4oyfKYPOO3hjLw94l', output=W, quiet=True)
# parche numpy 2.0 (WASB usa np.Inf/np.NaN)
subprocess.run(r"grep -rl 'np\.Inf' WASB-SBDT/src | xargs -r sed -i 's/np\.Inf/np.inf/g'", shell=True)
subprocess.run(r"grep -rl 'np\.NaN' WASB-SBDT/src | xargs -r sed -i 's/np\.NaN/np.nan/g'", shell=True)
print('setup OK | peso:', os.path.exists(W))

## 2) Encontrar el video + crear el meta (brasil: source_fps 24, stride 2)

In [ ]:
import json
mp4s=sorted(glob.glob('/kaggle/input/**/*.mp4', recursive=True), key=os.path.getsize, reverse=True)
assert mp4s, 'FALTA el video: Add Input -> subi el .mp4 de brasil.'
VIDEO=mp4s[0]; print('video:', VIDEO)
META='/kaggle/working/brasil.meta.json'
json.dump({'source_fps':24, 'frame_stride':2}, open(META,'w'))   # valores del tracking de brasil
LABELS='/kaggle/working/ncf_event_tracker/events_model/dataset/ball_gt/brasil_noruega_ball_labels.csv'
print('labels:', os.path.exists(LABELS))

## 3) Auditoría (split, sin entrenar)

In [ ]:
!rm -rf /kaggle/working/exp_audit
!cd /kaggle/working/ncf_event_tracker && python3 events_model/wasb_sparse.py \
  --video "{VIDEO}" --labels "{LABELS}" --meta "{META}" \
  --out /kaggle/working/exp_audit --audit-only

## 4) Entrenar (512×288, 8 épocas). Mirá BEFORE vs épocas.

In [ ]:
!rm -rf /kaggle/working/exp_512
!cd /kaggle/working/ncf_event_tracker && python3 events_model/wasb_sparse.py \
  --video "{VIDEO}" --labels "{LABELS}" --meta "{META}" \
  --wasb-src /kaggle/working/WASB-SBDT/src \
  --checkpoint {W} \
  --out /kaggle/working/exp_512 \
  --epochs 8 --batch-size 4 --device cuda 2>&1 | tail -40

## 5) Ver / bajar resultados

In [ ]:
import json
m=json.load(open('/kaggle/working/exp_512/metrics.json'))
base=[h for h in m if h['epoch']==-1][0]; best=max(m, key=lambda h:h['acc@100'])
print(f"BASELINE (soccer preentrenado): acc@100 {base['acc@100']:.3f}  acc@50 {base['acc@50']:.3f}")
print(f"MEJOR epoca {best['epoch']}: acc@100 {best['acc@100']:.3f}  (precision_visible {best.get('precision_visible',0):.2f})")
print('SUBE' if best['acc@100']>base['acc@100']+0.02 else 'NO sube claro')
from google.colab import files  # (en Kaggle: descargar desde el panel Output)


En Kaggle no hay `files.download`: bajá `metrics.json` y `best.pth` desde el
panel **Output** (o Save Version). Pegame `metrics.json` (BEFORE + épocas) y decidimos
A (seguir WASB) o B (cerrarlo).